# Notebook 02 — Feature Engineering

**Research topic:** *A Machine Learning Approach to Predicting Retail Investor Dormancy on Digital Investment Platforms*

### Purpose
This notebook transforms the preprocessed data into a customer-level modelling dataset.

The feature-engineering stage creates behavioural measures from transactions occurring **before the prediction point**. The future transaction file is used only to define the target variable.

### Target
- `dormant = 1`: investor had transaction activity during the observation period but no transaction during the prediction period.
- `dormant = 0`: investor continued to transact during the prediction period.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

OUTPUT_DIR = Path("outputs")

CUTOFF_DATE = pd.Timestamp("2023-09-30")

observation = pd.read_csv(OUTPUT_DIR / "observation_transactions.csv")

prediction = pd.read_csv(OUTPUT_DIR / "prediction_transactions.csv")

customers = pd.read_csv(OUTPUT_DIR / "clean_customer_data.csv")

observation["date"] = pd.to_datetime(observation["date"])
prediction["date"] = pd.to_datetime(prediction["date"])

print("Observation data:", observation.shape)
print("Prediction data:", prediction.shape)
print("Customer data:", customers.shape)


Observation data: (2367794, 4)
Prediction data: (791261, 4)
Customer data: (48723, 54)


In [2]:
# Basic checks before feature engineering
print("Observation customers:", observation["customer_id"].nunique())

print("Prediction-period customers:", prediction["customer_id"].nunique())

print("Customer records:", customers["customer_id"].nunique())


Observation customers: 48723
Prediction-period customers: 47943
Customer records: 48723


In [3]:
# Customer-level transaction features
features = observation.groupby("customer_id").agg(
    pre_tx_count=("amount", "size"),
    pre_total_volume=("amount", "sum"),
    pre_avg_tx_value=("amount", "mean"),
    pre_median_tx_value=("amount", "median"),
    pre_std_tx_value=("amount", "std"),
    pre_max_tx_value=("amount", "max"),
    pre_active_days=("date", "nunique"),
    pre_first_tx=("date", "min"),
    pre_last_tx=("date", "max"),).reset_index()

# A customer with only one transaction has no standard deviation.
features["pre_std_tx_value"] = (features["pre_std_tx_value"].fillna(0))

display(features.head())


,customer_id,pre_tx_count,pre_total_volume,pre_avg_tx_value,pre_median_tx_value,pre_std_tx_value,pre_max_tx_value,pre_active_days,pre_first_tx,pre_last_tx
0,10000617,6,39327200,6.554533e+06,5697300.0,2.926642e+06,11043950,6,2023-01-14,2023-08-15
1,10003212,7,18143600,2.591943e+06,2118600.0,1.480045e+06,5076800,7,2023-01-18,2023-07-04
2,10004554,6,24132250,4.022042e+06,3922850.0,2.553595e+06,7478700,6,2023-04-10,2023-08-10
3,10008981,11,14959600,1.359964e+06,903450.0,1.108231e+06,4078950,11,2023-02-10,2023-09-19
4,10011329,16,10084950,6.303094e+05,508600.0,5.507859e+05,2579750,15,2023-01-14,2023-09-30


In [4]:
# Createing transaction-type counts

type_counts = pd.crosstab( observation["customer_id"], observation["type"])

type_counts.columns = [ f"pre_{str(col).lower()}_count"
    for col in type_counts.columns]

type_counts = type_counts.reset_index()

features = features.merge(type_counts, on="customer_id", how="left")

# Ensure expected transaction-type features exist
for col in [
    "pre_deposit_count",
    "pre_withdrawal_count",
    "pre_payment_count",
    "pre_transfer_count",]:
    if col not in features.columns:
        features[col] = 0

display(features.head())


,customer_id,pre_tx_count,pre_total_volume,pre_avg_tx_value,pre_median_tx_value,pre_std_tx_value,pre_max_tx_value,pre_active_days,pre_first_tx,pre_last_tx,pre_deposit_count,pre_payment_count,pre_transfer_count,pre_withdrawal_count
0,10000617,6,39327200,6.554533e+06,5697300.0,2.926642e+06,11043950,6,2023-01-14,2023-08-15,0,1,5,0
1,10003212,7,18143600,2.591943e+06,2118600.0,1.480045e+06,5076800,7,2023-01-18,2023-07-04,0,2,5,0
2,10004554,6,24132250,4.022042e+06,3922850.0,2.553595e+06,7478700,6,2023-04-10,2023-08-10,1,2,2,1
3,10008981,11,14959600,1.359964e+06,903450.0,1.108231e+06,4078950,11,2023-02-10,2023-09-19,0,1,8,2
4,10011329,16,10084950,6.303094e+05,508600.0,5.507859e+05,2579750,15,2023-01-14,2023-09-30,1,6,7,2


In [6]:
# Behavioural features
# Recency: number of days since the investor's most recent

# observation-period transaction.
features["pre_recency_days"] = (CUTOFF_DATE - features["pre_last_tx"]).dt.days

# Transaction intensity.
features["pre_tx_per_active_day"] = (features["pre_tx_count"]/ features["pre_active_days"].replace(0, np.nan))

# Transaction-type ratios.
features["pre_deposit_ratio"] = (features["pre_deposit_count"]/ features["pre_tx_count"])

features["pre_withdrawal_ratio"] = (features["pre_withdrawal_count"]/ features["pre_tx_count"])

features["pre_payment_ratio"] = (features["pre_payment_count"] / features["pre_tx_count"])

features["pre_transfer_ratio"] = (features["pre_transfer_count"] / features["pre_tx_count"])

display(features.head())


,customer_id,pre_tx_count,pre_total_volume,pre_avg_tx_value,pre_median_tx_value,pre_std_tx_value,pre_max_tx_value,pre_active_days,pre_first_tx,pre_last_tx,pre_deposit_count,pre_payment_count,pre_transfer_count,pre_withdrawal_count,pre_recency_days,pre_tx_per_active_day,pre_deposit_ratio,pre_withdrawal_ratio,pre_payment_ratio,pre_transfer_ratio
0,10000617,6,39327200,6.554533e+06,5697300.0,2.926642e+06,11043950,6,2023-01-14,2023-08-15,0,1,5,0,46,1.000000,0.000000,0.000000,0.166667,0.833333
1,10003212,7,18143600,2.591943e+06,2118600.0,1.480045e+06,5076800,7,2023-01-18,2023-07-04,0,2,5,0,88,1.000000,0.000000,0.000000,0.285714,0.714286
2,10004554,6,24132250,4.022042e+06,3922850.0,2.553595e+06,7478700,6,2023-04-10,2023-08-10,1,2,2,1,51,1.000000,0.166667,0.166667,0.333333,0.333333
3,10008981,11,14959600,1.359964e+06,903450.0,1.108231e+06,4078950,11,2023-02-10,2023-09-19,0,1,8,2,11,1.000000,0.000000,0.181818,0.090909,0.727273
4,10011329,16,10084950,6.303094e+05,508600.0,5.507859e+05,2579750,15,2023-01-14,2023-09-30,1,6,7,2,0,1.066667,0.062500,0.125000,0.375000,0.437500


In [7]:
# Creating the dormancy target from future activity.
# IMPORTANT: prediction-period transactions are used only to create y.

future_customers = set(prediction["customer_id"].unique())

features["dormant"] = ( ~features["customer_id"].isin(future_customers)).astype(int)

print("Dormancy counts:")
display(features["dormant"].value_counts())

print("\nDormancy proportions:")
display(features["dormant"].value_counts(normalize=True))


Dormancy counts:


dormant
0    47943
1      780
Name: count, dtype: int64


Dormancy proportions:


dormant
0    0.983991
1    0.016009
Name: proportion, dtype: float64

In [8]:
# Meringe customer-level characteristics
customer_columns = [
    "customer_id",
    "age",
    "gender",
    "location",
    "income_bracket",
    "occupation",
    "education_level",
    "marital_status",
    "household_size",
    "acquisition_channel",
    "savings_account",
    "credit_card",
    "personal_loan",
    "investment_account",
    "insurance_product",]

available_customer_columns = [
    col for col in customer_columns
    if col in customers.columns]

modelling_data = features.merge(
    customers[available_customer_columns],
    on="customer_id",
    how="left")

print("Modelling dataset shape:", modelling_data.shape)
display(modelling_data.head())


Modelling dataset shape: (48723, 35)


,customer_id,pre_tx_count,pre_total_volume,pre_avg_tx_value,pre_median_tx_value,pre_std_tx_value,pre_max_tx_value,pre_active_days,pre_first_tx,pre_last_tx,pre_deposit_count,pre_payment_count,pre_transfer_count,pre_withdrawal_count,pre_recency_days,pre_tx_per_active_day,pre_deposit_ratio,pre_withdrawal_ratio,pre_payment_ratio,pre_transfer_ratio,dormant,age,gender,location,income_bracket,occupation,education_level,marital_status,household_size,acquisition_channel,savings_account,credit_card,personal_loan,investment_account,insurance_product
0,10000617,6,39327200,6.554533e+06,5697300.0,2.926642e+06,11043950,6,2023-01-14,2023-08-15,0,1,5,0,46,1.000000,0.000000,0.000000,0.166667,0.833333,0,44,Other,"Santa Marta, Magdalena",Low,Pharmacist,High School,Single,1,Referral,False,True,True,False,False
1,10003212,7,18143600,2.591943e+06,2118600.0,1.480045e+06,5076800,7,2023-01-18,2023-07-04,0,2,5,0,88,1.000000,0.000000,0.000000,0.285714,0.714286,0,45,Female,"Cali, Valle del Cauca",High,Bank Teller,High School,Divorced,4,Paid Ad,True,True,True,False,False
2,10004554,6,24132250,4.022042e+06,3922850.0,2.553595e+06,7478700,6,2023-04-10,2023-08-10,1,2,2,1,51,1.000000,0.166667,0.166667,0.333333,0.333333,0,45,Female,"Neiva, Huila",Medium,Musician,Bachelor,Divorced,2,Referral,True,True,False,True,False
3,10008981,11,14959600,1.359964e+06,903450.0,1.108231e+06,4078950,11,2023-02-10,2023-09-19,0,1,8,2,11,1.000000,0.000000,0.181818,0.090909,0.727273,0,60,Female,"Bogotá, Cundinamarca",Low,Waiter,High School,Married,4,Paid Ad,False,False,True,False,False
4,10011329,16,10084950,6.303094e+05,508600.0,5.507859e+05,2579750,15,2023-01-14,2023-09-30,1,6,7,2,0,1.066667,0.062500,0.125000,0.375000,0.437500,0,60,Female,"Bogotá, Cundinamarca",High,Hairdresser,Bachelor,Married,4,Organic,True,True,False,False,True


In [9]:
from sklearn.preprocessing import StandardScaler

# Selected monetary variables to be standardised
monetary_columns = [
    "pre_total_volume",
    "pre_avg_tx_value",
    "pre_median_tx_value",
    "pre_std_tx_value",
    "pre_max_tx_value",
]

# Standardise the monetary variables
scaler = StandardScaler()

for col in monetary_columns:
    if col in modelling_data.columns:
        modelling_data[f"std_{col}"] = scaler.fit_transform(
            modelling_data[[col]]
        ).ravel()

# Remove raw datetime columns from the modelling dataset
modelling_data = modelling_data.drop(
    columns=["pre_first_tx", "pre_last_tx"],
    errors="ignore"
)

display(modelling_data.head())

,customer_id,pre_tx_count,pre_total_volume,pre_avg_tx_value,pre_median_tx_value,pre_std_tx_value,pre_max_tx_value,pre_active_days,pre_deposit_count,pre_payment_count,pre_transfer_count,pre_withdrawal_count,pre_recency_days,pre_tx_per_active_day,pre_deposit_ratio,pre_withdrawal_ratio,pre_payment_ratio,pre_transfer_ratio,dormant,age,gender,location,income_bracket,occupation,education_level,marital_status,household_size,acquisition_channel,savings_account,credit_card,personal_loan,investment_account,insurance_product,std_pre_total_volume,std_pre_avg_tx_value,std_pre_median_tx_value,std_pre_std_tx_value,std_pre_max_tx_value
0,10000617,6,39327200,6.554533e+06,5697300.0,2.926642e+06,11043950,6,0,1,5,0,46,1.000000,0.000000,0.000000,0.166667,0.833333,0,44,Other,"Santa Marta, Magdalena",Low,Pharmacist,High School,Single,1,Referral,False,True,True,False,False,-0.033240,0.713879,0.940040,0.017339,-0.051175
1,10003212,7,18143600,2.591943e+06,2118600.0,1.480045e+06,5076800,7,0,2,5,0,88,1.000000,0.000000,0.000000,0.285714,0.714286,0,45,Female,"Cali, Valle del Cauca",High,Bank Teller,High School,Divorced,4,Paid Ad,True,True,True,False,False,-0.037735,-0.252322,-0.234687,-0.325324,-0.362097
2,10004554,6,24132250,4.022042e+06,3922850.0,2.553595e+06,7478700,6,1,2,2,1,51,1.000000,0.166667,0.166667,0.333333,0.333333,0,45,Female,"Neiva, Huila",Medium,Musician,Bachelor,Divorced,2,Referral,True,True,False,True,False,-0.036464,0.096380,0.357568,-0.071027,-0.236945
3,10008981,11,14959600,1.359964e+06,903450.0,1.108231e+06,4078950,11,0,1,8,2,11,1.000000,0.000000,0.181818,0.090909,0.727273,0,60,Female,"Bogotá, Cundinamarca",Low,Waiter,High School,Married,4,Paid Ad,False,False,True,False,False,-0.038410,-0.552717,-0.633566,-0.413397,-0.414090
4,10011329,16,10084950,6.303094e+05,508600.0,5.507859e+05,2579750,15,1,6,7,2,0,1.066667,0.062500,0.125000,0.375000,0.437500,0,60,Female,"Bogotá, Cundinamarca",High,Hairdresser,Bachelor,Married,4,Organic,True,True,False,False,True,-0.039444,-0.730629,-0.763178,-0.545442,-0.492207


In [10]:
# Checking for missing values after feature engineering
missing = (modelling_data.isna().sum().sort_values(ascending=False))

display(missing[missing > 0])


Series([], dtype: int64)

In [11]:
# Final target distribution
target_summary = pd.DataFrame({
    "Count": modelling_data["dormant"].value_counts(),
    "Percentage": (modelling_data["dormant"]
        .value_counts(normalize=True).mul(100))})

target_summary.index = [
    "Non-dormant (0)" if i == 0 else "Dormant (1)"
    for i in target_summary.index]

display(target_summary)


,Count,Percentage
Non-dormant (0),47943,98.399113
Dormant (1),780,1.600887


In [12]:
# Saving the final modelling dataset.
output_file = OUTPUT_DIR / "dormancy_modelling_dataset.csv"

modelling_data.to_csv(output_file, index=False)

print(f"Saved feature-engineered dataset to: {output_file}")
print(f"Rows: {len(modelling_data):,}")
print(f"Columns: {len(modelling_data.columns):,}")


Saved feature-engineered dataset to: outputs\dormancy_modelling_dataset.csv
Rows: 48,723
Columns: 38


## Feature-engineering outcome

The final `dormancy_modelling_dataset.csv` contains one row per investor.

It contains:
- historical transaction behaviour,
- transaction frequency and monetary features,
- transaction-type features,
- recency and activity-intensity measures,
- customer characteristics and
- the binary `dormant` target.

This dataset is now ready for the Logistic Regression and Gradient Boosting modelling.
